# Random Forest with Morgan Count Fingerprints — Chirality Baseline

This notebook establishes a baseline for understanding how molecular featurization affects a model's ability to predict chirality-related labels of increasing chemical difficulty.

Train a Random Forest on Morgan count fingerprints under two conditions:
- `includeChirality=True` — stereo information is encoded into the fingerprint
- `includeChirality=False` — **negative control**: fingerprint is chirality-blind; performance should collapse to ~50% chance

Evaluate on three target labels of increasing chemical depth:

| Target | Description | Difficulty |
|---|---|---|
| `R/S_class` | CIP absolute configuration | Easy — directly hashed into chiral Morgan FP|
| `@/@@_class` | SMILES parity tag | Harder —  these are from the arbitrary order of the SMILES string|
| `F/L_class` | Chromatographic elution order | Hardest — measures consequence of chirality in 3D space |

## Key finding from data inspection (see notebook 0)

`R/S` and `@/@@` are **nearly uncorrelated** in this dataset (Cramer's V ≈ 0.02 and Cohen's κ ≈ 0.02. See the correlation analysis in notebook 0.). 
- This is chemically meaningful: the `@`/`@@` SMILES parity tag depends on the arbitrary order in which atoms are written in the SMILES string, while **CIP R/S is determined by atomic priority rules that are resolved with RDKit converts a SMILES string to an RDKit object and resolves them into global, molecule-wide Cahn-Ingold-Prelog priority designations ((R), (S), (E), (Z)) based on atomic numbers and whole-molecule connectivity**.
- `R/S` and `@/@@` encode fundamentally different information. Predicting `@/@@` is therefore closer to reading a SMILES formatting convention than understanding molecular chirality.

## Note on the reference paper

The paper used this as a baseline:

```
Morgan fingerprints were generated with the extended connectivity fingerprints algorithm implemented by the RDKit library (2024.09.1). The GetMorganGenerator method of the rdFingerprintGenerator module was used for count fingerprints (with GetCountFingerprint). The fingerprints were generated with the following parameters: 
```
- radius = 3 (number of iterations to grow the fingerprint). this is the default.
- countSimulation = False. this is the default.

- **includeChirality = True (chirality information is added to the generated fingerprint). Default value is false.**

- useBondTypes = True (bond types are included as a part of the default bond invariants). this is the default.
- onlyNonzeroInvariants = False. this is the default.
- includeRingMembership = True. this is the default.
- countBounds = None (no boundaries for count simulation). this is the default.

- **fpSize = 512 (size of the generated fingerprint). Default is 2048**

- bondInvariantsGenerator = None. this is the default.
- atomInvariantsGenerator = None. this is the default.


In [1]:
import time

import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    matthews_corrcoef,
    accuracy_score,
    f1_score,
    average_precision_score,
)

# Load the data and splits

In [2]:
# load stereoisomer data
# https://github.com/jairesdesousa/chiraldlsv/blob/main/TSNE_maps/class_all.csv
input_path = "class_all.csv"
df = pd.read_csv(input_path, index_col=0)
df

,SMILES,SMILES_opp,TR/TE,F/L_class,@/@@_class,R/S_class
0,Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2,Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2,TE,F,@,S
1,Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2,Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2,TE,L,@@,R
2,C#CCO[C@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc1...,C#CCO[C@@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc...,TE,F,@,S
3,C#CCO[C@@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc...,C#CCO[C@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc1...,TE,L,@@,R
4,C=C(C(C)=O)[C@@H](CC(=O)c1ccc(Br)cc1)C(=O)OCC,C=C(C(C)=O)[C@H](CC(=O)c1ccc(Br)cc1)C(=O)OCC,TE,F,@@,R
...,...,...,...,...,...,...
3853,OC[C@]1(CCCOCc2ccccc2)COCCO1,OC[C@@]1(CCCOCc2ccccc2)COCCO1,TR,L,@,R
3854,OCCCC[C@H](O)c1ccc(F)cc1,OCCCC[C@@H](O)c1ccc(F)cc1,TR,F,@,S
3855,OCCCC[C@@H](O)c1ccc(F)cc1,OCCCC[C@H](O)c1ccc(F)cc1,TR,L,@@,R
3856,OCCCC[C@]1(CO)COCCO1,OCCCC[C@@]1(CO)COCCO1,TR,L,@,S


In [3]:
# this is the train and test set indices from the paper
df['TR/TE'].value_counts()

TR/TE
TR    3470
TE     388
Name: count, dtype: int64

In [4]:
df['F/L_class'].value_counts()

F/L_class
F    1929
L    1929
Name: count, dtype: int64

In [5]:
df['@/@@_class'].value_counts()

@/@@_class
@     1929
@@    1929
Name: count, dtype: int64

In [6]:
df['R/S_class'].value_counts()

R/S_class
S    1929
R    1929
Name: count, dtype: int64

In [7]:
def load_folds(path: str) -> list[dict[str, np.ndarray]]:
    """
    Load folds from a .npz file.

    Args:
        path: Filepath to a .npz file saved by save_folds().

    Returns:
        A list of dicts with keys "train", "val", and "test", matching
        the output format of make_kfold_splits().
    """
    archive = np.load(path)
    
    # Infer n_folds from the keys
    fold_indices = sorted(set(
        int(k.split("_")[0].replace("fold", ""))
        for k in archive.files
    ))
    
    return [
        {
            "train": archive[f"fold{i}_train"],
            "val":   archive[f"fold{i}_val"],
            "test":  archive[f"fold{i}_test"],
        }
        for i in fold_indices
    ]

In [8]:
mol_folds = load_folds("cmrt_folds.npz")
print(f"Loaded {len(mol_folds)} folds")

Loaded 5 folds


# Create features

In [9]:
def rdkit_to_np(vect, num_bits) -> np.ndarray:
    """
    Convert a sparse RDKit fingerprint vector to a dense float64 numpy array.
    Works for either rdkit.DataStructs.cDataStructs.ExplicitBitVect
    or rdkit.DataStructs.cDataStructs.UIntSparseIntVect.

    Using float64 (rather than the default uint32 from GetCountFingerprintAsNumPy)
    prevents integer overflow when fingerprints are subtracted to form reaction
    difference vectors.

    Args:
        vect: An RDKit ExplicitBitVect or UIntSparseIntVect fingerprint.
        num_bits: Length of the fingerprint (must match fpSize used to generate vect).

    Returns:
        Dense 1D numpy array of shape (num_bits,) and dtype float64.
    """
    arr = np.zeros((num_bits,), dtype=np.float64)
    DataStructs.ConvertToNumpyArray(vect, arr)  # overwrites arr in-place
    return arr

In [10]:
def calc_morgan_fp(
    smi: str,
    count: bool = True,
    radius: int = 2,
    fpSize: int = 2048,
    includeChirality: bool = True,
    useFeatures: bool = False,
) -> np.ndarray:
    """Compute a Morgan (ECFP-style) fingerprint for a SMILES string."""
    mol = Chem.MolFromSmiles(smi)
    atomInvariantsGenerator = (
        rdFingerprintGenerator.GetMorganFeatureAtomInvGen() if useFeatures else None
    )
    
    morgan_gen = rdFingerprintGenerator.GetMorganGenerator(
        radius=radius,
        fpSize=fpSize,
        includeChirality=includeChirality,
        atomInvariantsGenerator=atomInvariantsGenerator,
    )
    fp = getattr(morgan_gen,
                 f'Get{"Count" if count else ""}Fingerprint'
                 )(mol)

    return rdkit_to_np(fp, fpSize)

# Define target variables

All three targets are binary and perfectly balanced (50/50) by dataset construction — every molecule appears alongside its enantiomer, so every class has an equal counterpart.

**Chance baseline for all three targets: 50% accuracy.**

The prediction target could be
- `R/S_class`: CIP absolute chirality configuration
- `@/@@_class`: whether the SMILES string contains @ or @@
- `F/L_class`: whether a molecule is the **F**irst or **L**ast eluting enantiomer under chiral chromatography conditions. 

Note: `R/S_class` and `@/@@_class` encode related but distinct information (CIP stereodescriptors vs. chromatographic elution order). They are not used as targets here.

In [11]:
# Encode each binary target as 0/1
TARGET_LABEL_MAPS = {
    "R/S_class":  {"R": 0, "S": 1},
    "@/@@_class": {"@": 0, "@@": 1},
    "F/L_class":  {"F": 0, "L": 1},
}

for col, label_map in TARGET_LABEL_MAPS.items():
    df[f"label_{col}"] = df[col].map(label_map)

print("Class distributions (all should be 50/50):")
for col in TARGET_LABEL_MAPS:
    vc = df[f"label_{col}"].value_counts(normalize=True)
    print(f"  {col}: {vc.to_dict()}")

Class distributions (all should be 50/50):
  R/S_class: {1: 0.5, 0: 0.5}
  @/@@_class: {0: 0.5, 1: 0.5}
  F/L_class: {0: 0.5, 1: 0.5}


# Experimental conditions and hyperparameters

In [12]:
RF_HYPERPARAMETERS = {
    'n_estimators': 100,
    'max_depth': 25,
    'random_state': 42
}

MORGAN_BASE_PARAMS = {
    "count": True,
    "radius": 3,         # paper uses 3 (RDKit default is also 3)
    "fpSize": 512,       # paper uses 512; RDKit default is 2048
    # "includeChirality": True,  # critical for chirality prediction
    "useFeatures": False,
}

# two featurization conditions
FEATURIZATION_CONDITIONS = {
    "morgan_count_chiral":    {**MORGAN_BASE_PARAMS, "includeChirality": True},
    "morgan_count_no_chiral": {**MORGAN_BASE_PARAMS, "includeChirality": False},  # negative control
}

# Three prediction targets of increasing chemical difficulty
TARGETS = list(TARGET_LABEL_MAPS.keys())  # ['R/S_class', '@/@@_class', 'F/L_class']

# Training

- Outer loop: featurization condition (`includeChirality` True/False)  
- Middle loop: prediction target (`R/S`, `@/@@`, `F/L`)  
- Inner loop: fold (5 independent train/test splits)
**Total runs: 2 conditions × 3 targets × 5 folds = 30 model fits.**

For each fold:
1. Featurize train and test molecules using Morgan count fingerprints
2. Train a `RandomForestClassifier` on the training set
3. Evaluate on the held-out test set

**Note on validation set:** Random forests do not use a validation set during training (no early stopping or gradient-based optimization). The validation split is defined in the folds for consistency with other models (e.g., Chemprop, ChemBERTa) that do use it. However, the val split is intentionally not used here.


# Evaluation metrics

Compute 5 metrics per fold. 

---

### Metrics that require **predicted probabilities**

**AUROC (Area Under the ROC Curve)**  
- Measures the model's ability to *rank* positive examples above negative ones across all possible classification thresholds.
- A value of 1.0 means perfect ranking; 0.5 is chance.
- Because it is threshold-independent, it is the standard go-to for comparing classifiers across datasets. It requires probability scores (not hard labels) because it sweeps the threshold from 0 to 1 to construct the ROC curve.

**AUPRC (Area Under the Precision-Recall Curve)**  
- Summarizes the precision-recall tradeoff across thresholds.
- It is most informative when classes are imbalanced (where AUROC can be over-optimistic), but is still a useful complement at 50/50 balance.
- Like AUROC, it requires probability scores to construct the curve. A random classifier at 50/50 balance achieves an AUPRC of ~0.5.

---

### Metrics that require **hard class predictions** (binarized at threshold 0.5)

**Accuracy**  
- The fraction of all predictions that are correct: `(TP + TN) / total`.
- Interpretable and meaningful here because the classes are exactly balanced — the majority-class baseline is 50%, so any score above that reflects genuine learning.
- At imbalanced ratios, accuracy becomes misleading (a model predicting the majority class always can score 90%+ with zero predictive power).

**F1 Score**  
- The harmonic mean of precision and recall: `2 * (precision * recall) / (precision + recall)`.
- Penalizes both false positives and false negatives.
- At 50/50 balance it is numerically close to accuracy, but it is a better habit to report for binary classification since it becomes more informative at imbalance.
- Requires hard predictions because precision and recall are defined on discrete TP/FP/FN counts.

**MCC (Matthews Correlation Coefficient)**  
- Ranges from -1 (perfectly wrong) to +1 (perfectly correct), with 0 representing chance.
- It uses all four cells of the confusion matrix (TP, TN, FP, FN) and is considered the single most informative scalar metric for binary classification, particularly recommended by Pat Walters and Chicco & Jurman (2020).
- Unlike F1, it is symmetric with respect to both classes and is robust across different class distributions. Requires hard predictions.

In [13]:
def evaluate(
    model: RandomForestClassifier,
    X_test: np.ndarray,
    y_test: np.ndarray,
) -> dict:
    """Compute all evaluation metrics for a single fold's test set.

    Args:
        model: A fitted RandomForestClassifier.
        X_test: Feature matrix for the test set, shape (n_samples, n_features).
        y_test: True binary labels for the test set, shape (n_samples,).

    Returns:
        Dict mapping metric name to scalar value.
    """
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    return {
        "AUROC":    roc_auc_score(y_test, y_pred_proba),
        "MCC":      matthews_corrcoef(y_test, y_pred),
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1":       f1_score(y_test, y_pred),
        "AUPRC":    average_precision_score(y_test, y_pred_proba),
    }

In [14]:
all_results = []

for condition_name, morgan_params in FEATURIZATION_CONDITIONS.items():
    print(f"\n{'#' * 70}")
    print(f"Featurization: {condition_name}  (includeChirality={morgan_params['includeChirality']})")
    print(f"{'#' * 70}")

    for target_col in TARGETS:
        label_col = f"label_{target_col}"
        print(f"\n  Target: {target_col}")
        print(f"  {'-' * 60}")

        for fold_idx, fold in enumerate(mol_folds):
            START = time.time()
            print(f"{'=' * 60}")
            print(f"Fold {fold_idx}  |  train={len(fold['train'])}  val={len(fold['val'])}  test={len(fold['test'])}")
            print(f"{'=' * 60}")

            # slice dataframe using this fold's indices
            df_train = df.iloc[fold["train"]]
            df_test  = df.iloc[fold["test"]]
            # val intentionally unused for RF — see note above

            # Featurize
            X_train = np.stack(df_train["SMILES"].apply(calc_morgan_fp, **morgan_params).values)
            X_test  = np.stack(df_test["SMILES"].apply(calc_morgan_fp, **morgan_params).values)

            # get labels
            y_train = df_train[label_col].values
            y_test  = df_test[label_col].values

            # Train RF Classifier
            model = RandomForestClassifier(**RF_HYPERPARAMETERS)
            model.fit(X_train, y_train)

            # Predict and evaluate performance on test set
            metrics = evaluate(model, X_test, y_test)
            metrics["fold"]          = fold_idx
            metrics["model"]         = "RF"
            metrics["featurization"] = condition_name
            metrics["target"]        = target_col
            all_results.append(metrics)

            print(
                f"  fold {fold_idx} | "
                f"AUROC={metrics['AUROC']:.3f}  "
                f"MCC={metrics['MCC']:.3f}  "
                f"Acc={metrics['Accuracy']:.3f}  "
                f"F1={metrics['F1']:.3f}  "
                f"AUPRC={metrics['AUPRC']:.3f}"
            )
            print(f'  Elapsed time for fold {fold_idx}: {time.time() - START:.2f} seconds')


######################################################################
Featurization: morgan_count_chiral  (includeChirality=True)
######################################################################

  Target: R/S_class
  ------------------------------------------------------------
Fold 0  |  train=3478  val=190  test=190
  fold 0 | AUROC=0.940  MCC=0.675  Acc=0.837  F1=0.841  AUPRC=0.942
  Elapsed time for fold 0: 1.07 seconds
Fold 1  |  train=3478  val=190  test=190
  fold 1 | AUROC=0.939  MCC=0.705  Acc=0.853  F1=0.853  AUPRC=0.946
  Elapsed time for fold 1: 1.03 seconds
Fold 2  |  train=3478  val=190  test=190
  fold 2 | AUROC=0.950  MCC=0.717  Acc=0.858  F1=0.854  AUPRC=0.950
  Elapsed time for fold 2: 1.01 seconds
Fold 3  |  train=3478  val=190  test=190
  fold 3 | AUROC=0.934  MCC=0.663  Acc=0.832  F1=0.832  AUPRC=0.938
  Elapsed time for fold 3: 0.99 seconds
Fold 4  |  train=3478  val=190  test=190
  fold 4 | AUROC=0.958  MCC=0.705  Acc=0.853  F1=0.853  AUPRC=0.961
  Elapse

# Summarize performance across folds

Mean and standard deviation across the 5 test sets. These 5 scores per metric are what will be used in the Tukey HSD test when comparing this model against other featurization/model combinations.

In [15]:
print("Performance summary (RF + Morgan Count Fingerprints with Chirality)")
results_df = pd.DataFrame(all_results)
metric_cols = ["AUROC", "MCC", "Accuracy", "F1", "AUPRC"]

summary = (
    results_df
    .groupby(["featurization", "target"])[metric_cols]
    .agg(["mean", "std"])
    .round(4)
)

# Order targets by difficulty for readability
target_order = ["R/S_class", "@/@@_class", "F/L_class"]
summary = summary.reindex([(c, t) for c in FEATURIZATION_CONDITIONS for t in target_order])

print("Mean ± std across 5 folds (grouped by featurization and target):")
print(summary.to_string())

Performance summary (RF + Morgan Count Fingerprints with Chirality)
Mean ± std across 5 folds (grouped by featurization and target):
                                    AUROC             MCC         Accuracy              F1           AUPRC        
                                     mean     std    mean     std     mean     std    mean     std    mean     std
featurization          target                                                                                     
morgan_count_chiral    R/S_class   0.9442  0.0097  0.6930  0.0229   0.8463  0.0114  0.8464  0.0098  0.9473  0.0087
                       @/@@_class  0.8029  0.0338  0.4200  0.0720   0.7095  0.0364  0.7180  0.0326  0.8107  0.0405
                       F/L_class   0.8637  0.0140  0.5307  0.0368   0.7653  0.0185  0.7643  0.0218  0.8612  0.0142
morgan_count_no_chiral R/S_class   0.5000  0.0000  0.0000  0.0000   0.5000  0.0000  0.4876  0.0261  0.5000  0.0000
                       @/@@_class  0.5000  0.0000  0.0000  0.0

## Summary of results:
- Negative control worked as expected. All metrics show that the RF model is incapable of any predictive power beyond random chance when `includeChirality` is set to False for the fingerprint
    - A chirality-blind Morgan fingerprint produces a representation where enantiomers are literally identical vectors
    - Note: F1 shows a tiny non-zero std (±0.021–0.036) across all three targets. F1 is sensitive to the threshold and to which class the model arbitrarily assigns when features are tied, so minor implementation-level randomness can produce small fluctuations even when the model has zero discriminative power.
  
- Results show that predicting R/S is the easiest target, which is expected.
- However, initial hypothesis was that predicting F/L elution order would be more challenging than predicting @/@@
    - Predicting @/@@ is a genuinely difficult prediction task because the label is a SMILES artifact that doesn't align well with what Morgan fingerprints encode.
- **F/L is the most important scientific target and the AUROC of aroud 0.86 ndicates that Morgan count fingerprints carry meaningful signal.**

**Important differences relative to Table 1 from the paper**
- Table 1 from the paper reports a test set accuracy of 0.912 for predicting CIP R/S label. When using different splits, the results above show an average accuracy of only 0.84
    - Their code does use different splits (and [only 1 fold](https://github.com/jairesdesousa/chiraldlsv/blob/main/RF_models/RF_models.py#L27))
    - Their code allowed `max_depth` equal the default value of `None` [link](https://github.com/jairesdesousa/chiraldlsv/blob/main/RF_models/RF_models.py#L62) while I chose to regularize `max_depth=25` here
    - Their code used `random_state=0` while I used `random_state=42`
- Table 2 from the paper shows a test set accuracy of 0.716 for predicting the SMILES stereochemical label of @ vs. @@. which is inline with the 0.70 value shown above

## Sanity check: negative control

With `includeChirality=False`, the `@/@@` parity tag is removed from the fingerprint.
Since `R/S` and `@/@@` are near-independent (κ ≈ -0.02, see notebook 0), removing chirality encoding should collapse `@/@@` prediction to ~50%. `R/S` and `F/L` may retain some signal from molecular graph topology even without the parity tag.

In [16]:
METRIC = 'Accuracy'
print(f"{'Target':<15} {'Condition':<28} {'Mean Acc':>10} {'Check'}")
print("-" * 72)
for target in target_order:
    for condition in FEATURIZATION_CONDITIONS:
        subset = results_df[
            (results_df["target"] == target) &
            (results_df["featurization"] == condition)
        ]
        mean_acc = subset["Accuracy"].mean()
        is_control = "no_chiral" in condition
        flag = ""

        if is_control:
            flag = "✓ at chance" if mean_acc < 0.55 else "⚠ above chance — investigate"
        print(f"{target:<15} {condition:<28} {mean_acc:>10.4f} {flag}")
    print()

Target          Condition                      Mean Acc Check
------------------------------------------------------------------------
R/S_class       morgan_count_chiral              0.8463 
R/S_class       morgan_count_no_chiral           0.5000 ✓ at chance

@/@@_class      morgan_count_chiral              0.7095 
@/@@_class      morgan_count_no_chiral           0.5000 ✓ at chance

F/L_class       morgan_count_chiral              0.7653 
F/L_class       morgan_count_no_chiral           0.5000 ✓ at chance



# Save results

Save the per-fold scores to CSV so a future notebook can load all model + featurization results and run the Tukey HSD test in a single place via `scipy.stats.tukey_hsd` using the 5 per-fold scores as independent observations.

The `model`, `featurization`, and `target` columns identify each condition.

In [17]:
output_path = "results_RF_morgan_count.csv"
results_df.to_csv(output_path, index=False)
print(f"Saved {len(results_df)} rows to {output_path}")
print(f"  Conditions: {results_df['featurization'].unique().tolist()}")
print(f"  Targets:    {results_df['target'].unique().tolist()}")
print(f"  Folds:      {sorted(results_df['fold'].unique().tolist())}")
print(f"  Columns:    {results_df.columns.tolist()}")

Saved 30 rows to results_RF_morgan_count.csv
  Conditions: ['morgan_count_chiral', 'morgan_count_no_chiral']
  Targets:    ['R/S_class', '@/@@_class', 'F/L_class']
  Folds:      [0, 1, 2, 3, 4]
  Columns:    ['AUROC', 'MCC', 'Accuracy', 'F1', 'AUPRC', 'fold', 'model', 'featurization', 'target']
